In [1]:
import pandas as pd
import os
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

d:\Program\envs\agent_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
C:\Users\ANH TU\AppData\Local\Temp\ipykernel_16364\601060176.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
csv_file_path = r"D:\PharmaRAG-VN\Data\Clean\All_Documents_chunk.csv"
vector_db_path = r"D:\PharmaRAG-VN\Data\VectorStore\faiss_index"

In [3]:
df = pd.read_csv(csv_file_path)

In [4]:
df = df.fillna("")

In [5]:
documents = []

for _, row in df.iterrows():

    metadata = {
        "document_type": row["document_type"]
    }

    if row["level_1"]:
        metadata["level_1"] = row["level_1"]

    if row["level_2"]:
        metadata["level_2"] = row["level_2"]

    if row["level_3"]:
        metadata["level_3"] = row["level_3"]

    sections = []

    if row["document_type"]:
        sections.append(f"Document Type: {row['document_type']}")

    if row["level_1"]:
        sections.append(f"Level 1: {row['level_1']}")

    if row["level_2"]:
        sections.append(f"Level 2: {row['level_2']}")

    if row["level_3"]:
        sections.append(f"Level 3: {row['level_3']}")

    sections.append(f"Content:\n{row['content']}")

    embedding_text = "\n".join(sections)

    documents.append(
        Document(
            page_content=embedding_text,
            metadata=metadata
        )
    )

In [6]:
len(documents)

6856

In [7]:
documents[974].metadata

{'document_type': 'monograph',
 'level_1': '**BROMOCRIPTIN**',
 'level_2': '**Quá liều và bảo quản**'}

In [8]:
print(documents[974].page_content)

Document Type: monograph
Level 1: **BROMOCRIPTIN**
Level 2: **Quá liều và bảo quản**
Content:
### **Độ ổn định và bảo quản**  
Bảo quản ở nhiệt độ dưới 25 °C; đựng trong bao bì kín, tránh ánh sáng.  
### **Quá liều và xử trí**  
Quá liều bromocriptin có thể gây buồn nôn, nôn, hạ huyết áp thế đứng, vã mồ hôi và ảo giác. Xử trí quá liều bromocriptin bằng cách hút rửa dạ dày và truyền dịch tĩnh mạch để điều trị hạ huyết áp. Trường hợp có nôn và ảo giác, có thể chỉ định metoclopramid.


In [9]:
embeddings = OpenAIEmbeddings(
        model="text-embedding-3-large",
    )

In [10]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [11]:
os.makedirs(os.path.dirname(vector_db_path), exist_ok=True)
vectorstore.save_local(vector_db_path)